In [ ]:
# ============================================================
# SETUP — MOUNT DRIVE AND PREPARE CIFAR-10
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os

PROJECT = Path("/content/drive/MyDrive/Verifiable-Federated-Unlearning")
DATA_ROOT = Path("/content/cifar-10-python")
ARCHIVE = Path("/content/drive/MyDrive/cifar-10-python.tar.gz")

assert PROJECT.is_dir(), f"Project directory not found: {PROJECT}"
assert ARCHIVE.is_file(), f"CIFAR-10 archive not found: {ARCHIVE}"

# Extract only when the dataset is not already available.
if not (DATA_ROOT / "cifar-10-batches-py").is_dir():
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    !tar -xzf {ARCHIVE} -C {DATA_ROOT}

assert (DATA_ROOT / "cifar-10-batches-py").is_dir(), (
    f"CIFAR-10 dataset not found under {DATA_ROOT}"
)

os.environ["PYTHONPATH"] = (
    str(PROJECT) + os.pathsep + os.environ.get("PYTHONPATH", "")
)
os.environ["PYTHONUNBUFFERED"] = "1"

print("Setup complete")
print(f"Project: {PROJECT}")
print(f"Dataset: {DATA_ROOT}")


In [ ]:
# ============================================================
# TRAIN BACKDOORED MODEL AND CLEAN CONTROL
# ============================================================
SEED = 10

SCRIPT = PROJECT / "baeraser/scripts/train_backdoor.py"
OUT = PROJECT / f"baeraser/checkpoints/seed_{SEED}"
OUT.mkdir(parents=True, exist_ok=True)

assert SCRIPT.is_file(), f"Training script not found: {SCRIPT}"

def train_model(poison_rate, output_name):
    checkpoint = OUT / output_name

    print("\n" + "=" * 65, flush=True)
    print(f"Training seed: {SEED}", flush=True)
    print(f"Poison rate:   {poison_rate}", flush=True)
    print(f"Checkpoint:    {checkpoint}", flush=True)
    print("=" * 65, flush=True)

    # Direct, unbuffered execution: output streams into the notebook.
    !PYTHONPATH={PROJECT} python -u {SCRIPT} \
        --data {DATA_ROOT} \
        --out {checkpoint} \
        --epochs 15 \
        --batch-size 128 \
        --lr 0.01 \
        --poison-rate {poison_rate} \
        --trigger-size 6 \
        --seed {SEED} \
        --workers 2 \
        --progress-every 5

    if not checkpoint.is_file():
        raise FileNotFoundError(
            f"Training failed or checkpoint not found: {checkpoint}"
        )

    print(f"\nCheckpoint saved: {checkpoint}", flush=True)

train_model(0.10, "backdoored_model.pt")
train_model(0.0, "clean_model.pt")

print("\n" + "=" * 65, flush=True)
print(f"SEED {SEED}: BOTH MODELS TRAINED SUCCESSFULLY", flush=True)
print(f"Output directory: {OUT}", flush=True)
print("=" * 65, flush=True)


In [ ]:
# ============================================================
# RUN TRIGGER-RECOVERY VALIDATION
# ============================================================
SEED = 10

BACKDOORED_MODEL = (
    PROJECT / f"baeraser/checkpoints/seed_{SEED}/backdoored_model.pt"
)
CLEAN_MODEL = (
    PROJECT / f"baeraser/checkpoints/seed_{SEED}/clean_model.pt"
)
OUTPUT = PROJECT / f"baeraser/results/seed_{SEED}"

assert (PROJECT / "baeraser/recovery.py").is_file(), "recovery.py missing"
assert (PROJECT / "baeraser/tests/test_recovery.py").is_file(), (
    "test_recovery.py missing"
)
assert BACKDOORED_MODEL.is_file(), f"Missing: {BACKDOORED_MODEL}"
assert CLEAN_MODEL.is_file(), f"Missing: {CLEAN_MODEL}"
assert (DATA_ROOT / "cifar-10-batches-py").is_dir(), (
    f"CIFAR-10 dataset not found: {DATA_ROOT}"
)

OUTPUT.mkdir(parents=True, exist_ok=True)
os.environ["BAERASER_TEST_MODEL"] = str(BACKDOORED_MODEL)
os.environ["BAERASER_TEST_CLEAN_MODEL"] = str(CLEAN_MODEL)
os.environ["BAERASER_TEST_DATA_ROOT"] = str(DATA_ROOT)
os.environ["BAERASER_TEST_OUTPUT"] = str(OUTPUT)
os.environ["PYTHONUNBUFFERED"] = "1"

print(f"Validation seed: {SEED}", flush=True)
print(f"Backdoored model: {BACKDOORED_MODEL}", flush=True)
print(f"Clean control:    {CLEAN_MODEL}", flush=True)
print(f"Output directory: {OUTPUT}", flush=True)

# Run from the repository root and stream unittest output directly.
%cd /content/drive/MyDrive/Verifiable-Federated-Unlearning
!PYTHONPATH=/content/drive/MyDrive/Verifiable-Federated-Unlearning python -u -m unittest baeraser.tests.test_recovery -v

print(f"\nValidation command finished. Inspect results in: {OUTPUT}")
